# 🏠 Housing Price Prediction — Full Analysis
**Dataset:** Housing.csv &nbsp;|&nbsp; **Target:** `price` &nbsp;|&nbsp; **Models:** Linear Regression & Random Forest

---

## 📦 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries imported successfully!')

---
## Task 1 — Data Loading & Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('Housing.csv')

print('=== First 10 Rows ===')
df.head(10)

In [ ]:
# Shape
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')
print()

# Target vs Features
print('Target column  : price')
print('Feature columns:', [c for c in df.columns if c != 'price'])

In [ ]:
# Data types
df.dtypes

In [ ]:
# Missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()} — dataset is clean!')

---
## Task 2 — Data Cleaning

In [ ]:
# Step 1: Check and remove duplicates
print(f'Duplicate rows found: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Shape after deduplication: {df.shape}')

In [ ]:
# Step 2: Convert binary yes/no columns to 1/0
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

print('Binary encoding applied to:', binary_cols)

In [ ]:
# Step 3: One-hot encode furnishingstatus (3 categories → 2 dummy columns)
print('Value counts for furnishingstatus:')
print(df['furnishingstatus'].value_counts())

df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)
print('\nColumns after encoding:', list(df.columns))

In [ ]:
# Final cleaned dataset preview
df.head(5)

---
## Task 3 — Model Building

In [ ]:
# Split features and target
X = df.drop('price', axis=1)
y = df['price']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples: {X_train.shape[0]}  |  Test samples: {X_test.shape[0]}')

In [ ]:
# ── Model 1: Linear Regression ──
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_mae  = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2   = r2_score(y_test, y_pred_lr)

print('=== Linear Regression ===')
print(f'  MAE  : {lr_mae:,.0f}')
print(f'  RMSE : {lr_rmse:,.0f}')
print(f'  R²   : {lr_r2:.4f}')

In [ ]:
# ── Model 2: Random Forest Regressor ──
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2   = r2_score(y_test, y_pred_rf)

print('=== Random Forest ===')
print(f'  MAE  : {rf_mae:,.0f}')
print(f'  RMSE : {rf_rmse:,.0f}')
print(f'  R²   : {rf_r2:.4f}')

In [ ]:
# ── Model Comparison Table ──
comparison = pd.DataFrame({
    'Model':   ['Linear Regression', 'Random Forest'],
    'MAE':     [f'{lr_mae:,.0f}',  f'{rf_mae:,.0f}'],
    'RMSE':    [f'{lr_rmse:,.0f}', f'{rf_rmse:,.0f}'],
    'R² Score':[f'{lr_r2:.4f}',   f'{rf_r2:.4f}']
})
print('=== Model Comparison ===')
comparison

In [ ]:
# Feature Importances (Random Forest)
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Top Feature Importances (Random Forest):')
fi

---
## Task 4 — Visualizations

In [ ]:
BG   = '#F8F9FA'
DARK = '#2D3142'

In [ ]:
# ── Chart 1: Price Distribution Histogram ──
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(BG)

ax.hist(df['price'] / 1e6, bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(df['price'].mean() / 1e6,   color='#C44E52', lw=2, linestyle='--',
           label=f'Mean:   {df["price"].mean()/1e6:.2f}M')
ax.axvline(df['price'].median() / 1e6, color='#55A868', lw=2, linestyle='--',
           label=f'Median: {df["price"].median()/1e6:.2f}M')

ax.set_xlabel('House Price (in Millions)', fontsize=13, color=DARK)
ax.set_ylabel('Number of Houses', fontsize=13, color=DARK)
ax.set_title('Distribution of House Prices', fontsize=16, fontweight='bold', color=DARK, pad=15)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: Correlation Heatmap ──
fig, ax = plt.subplots(figsize=(11, 8), facecolor=BG)
ax.set_facecolor(BG)

corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9},
            vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Features vs Price',
             fontsize=16, fontweight='bold', color=DARK, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: Actual vs Predicted — Side-by-side for both models ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)

for ax, y_pred, model, r2, color in zip(
        axes,
        [y_pred_lr, y_pred_rf],
        ['Linear Regression', 'Random Forest'],
        [lr_r2, rf_r2],
        ['#4C72B0', '#DD8452']):

    ax.set_facecolor(BG)
    ax.scatter(y_test / 1e6, y_pred / 1e6,
               alpha=0.55, color=color, edgecolors='white', lw=0.4, s=60)

    mn = min(y_test.min(), y_pred.min()) / 1e6
    mx = max(y_test.max(), y_pred.max()) / 1e6
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect Fit')

    ax.set_xlabel('Actual Price (Millions)', fontsize=12, color=DARK)
    ax.set_ylabel('Predicted Price (Millions)', fontsize=12, color=DARK)
    ax.set_title(f'{model}\nR\u00b2 = {r2:.4f}', fontsize=13, fontweight='bold', color=DARK)
    ax.legend(fontsize=10)

fig.suptitle('Actual vs Predicted House Prices',
             fontsize=16, fontweight='bold', color=DARK, y=1.02)
plt.tight_layout()
plt.show()

---
## Task 5 — Insights & Summary

### Key Findings

**Which features influence house price the most?**  
The most influential feature by a large margin is **area** (plot size), which alone accounts for ~47% of the model's predictive power according to Random Forest feature importances. After area, **number of bathrooms** (15%) and **air conditioning** (6%) are the next strongest drivers of price. Amenities like a preferred area location, basement, and parking also contribute meaningfully.

**How accurate was the model (in plain terms)?**  
The **Linear Regression model performed slightly better** with an R² of **0.653**, meaning it explains about 65% of the variation in house prices. On average, its predictions were off by roughly **970,000 units** (MAE). For a dataset of 545 homes with prices ranging from 1.75M to 13.3M, this is a reasonable baseline — the model captures the broad trend well but misses finer nuances.

**What surprised you in the data?**  
Two things stood out: (1) **Random Forest did NOT outperform Linear Regression** — this is unusual and suggests the relationships in this dataset are largely linear, with no complex nonlinear patterns that Random Forest's extra power could exploit. (2) The dataset had **zero missing values and zero duplicates**, which is rare in real-world housing data and suggests it may be a curated/synthetic dataset.

**One recommendation for a real estate business:**  
Since **area is by far the strongest price predictor**, real estate agencies should prioritize square footage data quality in their listings and use it as the primary filter in automated valuation models. Additionally, highlighting **air conditioning** and **number of bathrooms** in listings can justify premium pricing — these features show the strongest correlation with price after area.